# Sección 1: Setup

Configuración inicial del notebook de generación del manifest del dataset GeoVision-CLIP Cali. Define imports, constantes globales y conexión al bucket de HuggingFace donde reside el panel de datos.

## 1.1 Imports

Importación de librerías necesarias para construir el manifest: manipulación de archivos y rutas (`os`, `pathlib`), serialización JSON, cálculo de hashes (`hashlib`), manejo de fechas, inspección de Zarr (`xarray`), datos tabulares (`pandas`), y acceso al bucket de HuggingFace.

In [1]:
import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
import warnings

import numpy as np
import pandas as pd
import xarray as xr

from huggingface_hub import (
    HfApi,
    HfFileSystem,
    list_bucket_tree,
    download_bucket_files,
)

warnings.filterwarnings('ignore')

print('Imports OK')

C:\Users\Administrador\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## 1.2 Configuración global

Constantes del manifest: identificador del dataset, versión, repositorio HuggingFace, BBox de Cali, rango temporal del proyecto y rutas de salida. El manifest se guarda localmente y luego se sube a Git.

In [2]:
HF_REPO_ID = 'yeigen/fuentes-proyecto-3'
HF_TOKEN = 'hf_qRVghKFsgDYRsqQIHcWBdbMoJV-TOtmefrn'

DATASET_ID = 'geovision_cali_v1'
VERSION = '1.0'

BBOX_PROYECTO = {
    'lon_min': -76.60,
    'lon_max': -76.40,
    'lat_min': 3.30,
    'lat_max': 3.55,
    'crs': 'EPSG:4326',
}

FECHA_INICIO_PROYECTO = '2021-01-01'
FECHA_FIN_PROYECTO = '2025-12-31'

RUTAS_FUENTES = {
    'S5P_NO2':   'copernicus_s5p_offl_l3_no2/panel.zarr',
    'S5P_O3':    'copernicus_s5p_offl_l3_o3/panel.zarr',
    'S5P_SO2':   'copernicus_s5p_offl_l3_so2/panel.zarr',
    'S2':        'copernicus_s2_sr_harmonized/panel.zarr',
    'ERA5':      'ecmwf_era5_hourly/panel.zarr',
    'MODIS_AOD': 'modis_061_mcd19a2_granules/panel.zarr',
}

DESCRIPCION_FUENTES = {
    'S5P_NO2':   {'name': 'Sentinel-5P L3 OFFL NO2',     'description': 'Tropospheric NO2 column density', 'units': 'mol/m^2'},
    'S5P_O3':    {'name': 'Sentinel-5P L3 OFFL O3',      'description': 'Total column O3 density',          'units': 'mol/m^2'},
    'S5P_SO2':   {'name': 'Sentinel-5P L3 OFFL SO2',     'description': 'SO2 column density',               'units': 'mol/m^2'},
    'S2':        {'name': 'Sentinel-2 L2A harmonized',   'description': 'Surface reflectance, 13 bands',    'units': 'reflectance'},
    'ERA5':      {'name': 'ECMWF ERA5-Land hourly',      'description': 'Meteorological reanalysis',        'units': 'mixed'},
    'MODIS_AOD': {'name': 'MODIS MCD19A2 MAIAC AOD',     'description': 'Aerosol Optical Depth (PM proxy)', 'units': 'unitless'},
}

DIR_OUTPUT = Path('./manifest_output')
DIR_OUTPUT.mkdir(exist_ok=True)
DIR_DAGMA = Path('./datos_dagma')
DIR_DAGMA.mkdir(exist_ok=True)

print(f'Dataset: {DATASET_ID} v{VERSION}')
print(f'Repositorio: {HF_REPO_ID}')
print(f'Fuentes Zarr: {len(RUTAS_FUENTES)}')
print(f'Output: {DIR_OUTPUT.resolve()}')

Dataset: geovision_cali_v1 v1.0
Repositorio: yeigen/fuentes-proyecto-3
Fuentes Zarr: 6
Output: C:\Users\Administrador\Desktop\github_proyecto03\proyecto-3\manifest\manifest_output


## 1.3 Conexión a HuggingFace y listado completo del bucket

Inicialización del cliente API, filesystem virtual del bucket y listado completo de archivos con sus tamaños y hashes (si están disponibles). Se almacena la información en una estructura que será procesada por fuente en las siguientes celdas.

In [3]:
api = HfApi(token=HF_TOKEN)
fs = HfFileSystem(token=HF_TOKEN)

print('Obteniendo árbol completo del bucket...')
items = list(list_bucket_tree(HF_REPO_ID, token=HF_TOKEN))
files_items = [it for it in items if it.type == 'file']

print(f'Total items: {len(items):,}')
print(f'Archivos: {len(files_items):,}')

attrs_disponibles = [a for a in dir(files_items[0]) if not a.startswith('_')]
print(f'\nAtributos disponibles en items: {attrs_disponibles}')

ejemplo = files_items[0]
print(f'\nEjemplo del primer archivo:')
for attr in attrs_disponibles:
    try:
        val = getattr(ejemplo, attr)
        if not callable(val):
            print(f'  {attr}: {val}')
    except Exception:
        pass

Obteniendo árbol completo del bucket...
Total items: 8,847
Archivos: 8,847

Atributos disponibles en items: ['mtime', 'path', 'size', 'type', 'uploaded_at', 'xet_hash']

Ejemplo del primer archivo:
  mtime: 2026-05-11 02:26:53.929000+00:00
  path: copernicus_s2_sr_harmonized/panel.zarr/.zattrs
  size: 2
  type: file
  uploaded_at: 2026-05-11 02:55:44.652000+00:00
  xet_hash: 56fc0e24abad6feca39134b95b46331aed808c02bf7190c3835be08448dce0ba


# Sección 2: Catálogo de archivos por fuente

## 2.1 Agrupación de archivos y cálculo de tamaños

Para cada fuente del dataset, se identifican sus archivos en el bucket, se suma el peso total, se cuenta el número de chunks y se intenta extraer el hash MD5 individual si está disponible en el listado del bucket.

In [4]:
def get_md5_attr(item):
    for attr in ['lfs_hash', 'md5', 'sha256', 'etag', 'oid']:
        if hasattr(item, attr):
            val = getattr(item, attr)
            if val and not callable(val):
                return str(val), attr
    return None, None

prefijos = {nombre: ruta.rstrip('/') + '/' for nombre, ruta in RUTAS_FUENTES.items()}
prefijos['DAGMA'] = 'dagma/'

catalogo = {nombre: {'archivos': [], 'size_bytes': 0, 'n_files': 0} for nombre in prefijos}

for item in files_items:
    ruta = item.path
    for nombre, prefijo in prefijos.items():
        if ruta.startswith(prefijo):
            md5_val, md5_source = get_md5_attr(item)
            catalogo[nombre]['archivos'].append({
                'ruta': ruta,
                'size_bytes': item.size,
                'md5': md5_val,
                'md5_source': md5_source,
            })
            catalogo[nombre]['size_bytes'] += item.size
            catalogo[nombre]['n_files'] += 1
            break

print(f"{'Fuente':<14s} | {'Archivos':>10s} | {'Tamaño':>12s} | MD5 disponible")
print('-' * 60)
for nombre, info in catalogo.items():
    size_gb = info['size_bytes'] / 1e9
    size_mb = info['size_bytes'] / 1e6
    size_str = f'{size_gb:.2f} GB' if size_gb >= 1 else f'{size_mb:.1f} MB'
    md5_disp = 'Sí' if (info['n_files'] > 0 and info['archivos'][0]['md5']) else 'No'
    print(f'{nombre:<14s} | {info["n_files"]:>10,d} | {size_str:>12s} | {md5_disp}')

total_bytes = sum(info['size_bytes'] for info in catalogo.values())
total_files = sum(info['n_files'] for info in catalogo.values())
print('-' * 60)
print(f'{"TOTAL":<14s} | {total_files:>10,d} | {total_bytes/1e9:>9.2f} GB')

Fuente         |   Archivos |       Tamaño | MD5 disponible
------------------------------------------------------------
S5P_NO2        |        216 |      20.0 MB | No
S5P_O3         |        152 |      14.8 MB | No
S5P_SO2        |        280 |       8.1 MB | No
S2             |      8,102 |     89.67 GB | No
ERA5           |         40 |       4.7 MB | No
MODIS_AOD      |         54 |      16.0 MB | No
DAGMA          |          3 |       0.9 MB | No
------------------------------------------------------------
TOTAL          |      8,847 |     89.73 GB


# Sección 3: Extracción de metadata por fuente

## 3.1 Funciones helper

Funciones reutilizables para: cargar un Zarr lazy desde el bucket, parsear el formato temporal específico de cada fuente, y calcular un hash agregado del catálogo de chunks (sin descargar los datos).

In [5]:
def cargar_zarr(ruta_zarr):
    mapper = fs.get_mapper(f'buckets/{HF_REPO_ID}/{ruta_zarr}')
    return xr.open_zarr(mapper)


def parsear_tiempo_fuente(time_values, source_name):
    s = [str(t) for t in time_values]
    if source_name.startswith('S5P') or source_name == 'S2':
        starts = [x.split('_')[0] for x in s]
        return pd.to_datetime(starts, format='%Y%m%dT%H%M%S')
    elif source_name == 'ERA5':
        return pd.to_datetime(s, format='%Y%m%dT%H')
    elif source_name == 'MODIS_AOD':
        return pd.to_datetime(s, format='A%Y%j')
    else:
        raise ValueError(f'Formato temporal no definido para {source_name}')


def hash_agregado_fuente(archivos):
    md5_global = hashlib.md5()
    for archivo in sorted(archivos, key=lambda x: x['ruta']):
        key = f"{archivo['ruta']}|{archivo['size_bytes']}|{archivo['md5'] or 'no-md5'}"
        md5_global.update(key.encode('utf-8'))
    return md5_global.hexdigest()


print('Funciones definidas: cargar_zarr, parsear_tiempo_fuente, hash_agregado_fuente')

Funciones definidas: cargar_zarr, parsear_tiempo_fuente, hash_agregado_fuente


## 3.2 Metadata de las fuentes Zarr

Para cada una de las 6 fuentes Zarr se carga el panel en modo lazy, se extraen dimensiones, bandas, BBox espacial, rango temporal y resolución. La información queda almacenada en `metadata_zarr` para ensamblado final.

In [6]:
metadata_zarr = {}

for nombre, ruta in RUTAS_FUENTES.items():
    print(f'Procesando {nombre}...')
    try:
        ds = cargar_zarr(ruta)
        tiempo = parsear_tiempo_fuente(ds.time.values, nombre)

        bands = [str(b) for b in ds.band.values] if 'band' in ds.coords else []

        res_deg = abs(float(ds.y[1] - ds.y[0])) if len(ds.y) > 1 else 0
        res_km = res_deg * 111

        metadata_zarr[nombre] = {
            'dimensiones': {dim: int(ds.sizes[dim]) for dim in ds.sizes},
            'variables': bands,
            'n_variables': len(bands),
            'spatial_bbox': [
                round(float(ds.x.min()), 6),
                round(float(ds.y.min()), 6),
                round(float(ds.x.max()), 6),
                round(float(ds.y.max()), 6),
            ],
            'spatial_resolution_deg': round(res_deg, 6),
            'spatial_resolution_km': round(res_km, 4),
            'temporal_range': [
                tiempo.min().isoformat(),
                tiempo.max().isoformat(),
            ],
            'n_observaciones': int(len(tiempo)),
            'time_dtype': str(ds.time.dtype),
        }
        print(f'  OK - {len(bands)} bandas, {len(tiempo):,} timestamps, {res_km:.2f} km')
    except Exception as e:
        print(f'  ERROR: {e}')
        metadata_zarr[nombre] = {'error': str(e)}

print(f'\nFuentes procesadas: {len(metadata_zarr)}')

Procesando S5P_NO2...
  OK - 3 bandas, 25,592 timestamps, 1.11 km
Procesando S5P_O3...
  OK - 2 bandas, 25,716 timestamps, 1.11 km
Procesando S5P_SO2...
  OK - 2 bandas, 25,829 timestamps, 1.11 km
Procesando S2...
  OK - 13 bandas, 1,552 timestamps, 0.01 km
Procesando ERA5...
  OK - 8 bandas, 43,824 timestamps, 27.75 km
Procesando MODIS_AOD...
  OK - 4 bandas, 1,826 timestamps, 0.92 km

Fuentes procesadas: 6


## 3.3 Metadata de DAGMA

Se descarga el `manifest_ground_truth.json` que Gabriel ya generó (contiene MD5 originales de los archivos DAGMA), se carga el Parquet de mediciones y el CSV de metadata de estaciones para extraer información complementaria (rango temporal real, contaminantes únicos, estaciones únicas, cobertura por combinación estación-contaminante).

In [7]:
download_bucket_files(
    HF_REPO_ID,
    files=[
        ('dagma/dagma_cvc_horario_raw.parquet', str(DIR_DAGMA / 'dagma_cvc_horario_raw.parquet')),
        ('dagma/estaciones_metadata.csv',       str(DIR_DAGMA / 'estaciones_metadata.csv')),
        ('dagma/manifest_ground_truth.json',    str(DIR_DAGMA / 'manifest_ground_truth.json')),
    ],
    token=HF_TOKEN,
)

with open(DIR_DAGMA / 'manifest_ground_truth.json') as f:
    manifest_dagma_original = json.load(f)

df_dagma = pd.read_parquet(DIR_DAGMA / 'dagma_cvc_horario_raw.parquet')
df_estaciones = pd.read_csv(DIR_DAGMA / 'estaciones_metadata.csv')

cobertura_dagma = (
    df_dagma.groupby(['nombre_est', 'msfl_code'])
    .size()
    .reset_index(name='n_mediciones')
    .to_dict('records')
)

metadata_dagma = {
    'fuente_original': manifest_dagma_original.get('fuente', 'DAGMA/CVC'),
    'dataset_origen': manifest_dagma_original.get('dataset_id_origen', None),
    'n_estaciones': int(df_estaciones.shape[0]),
    'contaminantes': sorted(df_dagma['msfl_code'].unique().tolist()),
    'temporal_range': [
        df_dagma['med_fecha_inicio'].min().isoformat(),
        df_dagma['med_fecha_inicio'].max().isoformat(),
    ],
    'n_mediciones_total': int(len(df_dagma)),
    'estaciones': df_estaciones.to_dict('records'),
    'cobertura_estacion_contaminante': cobertura_dagma,
    'unidades': 'µg/m^3',
    'manifest_original': manifest_dagma_original,
}

print(f'DAGMA procesado:')
print(f'  {metadata_dagma["n_estaciones"]} estaciones')
print(f'  {len(metadata_dagma["contaminantes"])} contaminantes: {metadata_dagma["contaminantes"]}')
print(f'  {metadata_dagma["n_mediciones_total"]:,} mediciones totales')
print(f'  Rango: {metadata_dagma["temporal_range"][0]} a {metadata_dagma["temporal_range"][1]}')

DAGMA procesado:
  10 estaciones
  3 contaminantes: ['NO2', 'O3', 'SO2']
  107,291 mediciones totales
  Rango: 2020-01-01T00:00:00 a 2024-12-31T23:00:00


# Sección 4: Ensamblaje y guardado del manifest

## 4.1 Construcción del diccionario completo

Combina toda la metadata recolectada en una sola estructura JSON. Para cada fuente Zarr se incluye: descripción, ruta, peso, hash agregado, número de archivos, dimensiones, bandas, rango espacial y temporal, resolución y unidades. DAGMA se agrega con su estructura específica.

In [8]:
def to_serializable(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (pd.Timestamp, datetime)):
        return obj.isoformat()
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_serializable(x) for x in obj]
    return obj


sources_dict = {}

for nombre in RUTAS_FUENTES:
    info_cat = catalogo[nombre]
    meta = metadata_zarr.get(nombre, {})
    desc = DESCRIPCION_FUENTES[nombre]

    sources_dict[nombre] = {
        'name': desc['name'],
        'description': desc['description'],
        'format': 'Zarr',
        'path': RUTAS_FUENTES[nombre],
        'units': desc['units'],
        'n_files': info_cat['n_files'],
        'size_bytes': info_cat['size_bytes'],
        'size_mb': round(info_cat['size_bytes'] / 1e6, 3),
        'aggregate_hash_md5': hash_agregado_fuente(info_cat['archivos']),
        **meta,
    }

dagma_cat = catalogo['DAGMA']
sources_dict['DAGMA'] = {
    'name': 'DAGMA / CVC ground truth',
    'description': 'In-situ hourly measurements from monitoring stations',
    'format': 'Parquet + CSV + JSON',
    'path': 'dagma/',
    'n_files': dagma_cat['n_files'],
    'size_bytes': dagma_cat['size_bytes'],
    'size_mb': round(dagma_cat['size_bytes'] / 1e6, 3),
    'aggregate_hash_md5': hash_agregado_fuente(dagma_cat['archivos']),
    **metadata_dagma,
}

total_bytes_all = sum(catalogo[n]['size_bytes'] for n in catalogo)
total_files_all = sum(catalogo[n]['n_files'] for n in catalogo)

manifest = {
    'dataset_id': DATASET_ID,
    'version': VERSION,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'creators': 'Equipo GeoVision-CLIP Cali — UAO',
    'project': 'GeoVision-CLIP Cali: Estimación de contaminación atmosférica',
    'repository': {
        'platform': 'HuggingFace Bucket',
        'id': HF_REPO_ID,
        'url': f'https://huggingface.co/datasets/{HF_REPO_ID}',
    },
    'spatial_extent': {
        'bbox': [BBOX_PROYECTO['lon_min'], BBOX_PROYECTO['lat_min'],
                 BBOX_PROYECTO['lon_max'], BBOX_PROYECTO['lat_max']],
        'crs': BBOX_PROYECTO['crs'],
        'region': 'Área metropolitana de Santiago de Cali, Valle del Cauca, Colombia',
    },
    'temporal_extent': {
        'start': FECHA_INICIO_PROYECTO,
        'end': FECHA_FIN_PROYECTO,
    },
    'summary': {
        'total_size_bytes': total_bytes_all,
        'total_size_gb': round(total_bytes_all / 1e9, 3),
        'total_files': total_files_all,
        'n_sources': len(sources_dict),
        'meets_50gb_threshold': total_bytes_all >= 50 * 1e9,
    },
    'sources': sources_dict,
}

manifest = to_serializable(manifest)

print('Manifest ensamblado:')
print(f'  Fuentes: {len(manifest["sources"])}')
print(f'  Peso total: {manifest["summary"]["total_size_gb"]} GB')
print(f'  Archivos: {manifest["summary"]["total_files"]:,}')
print(f'  Umbral ≥ 50 GB: {manifest["summary"]["meets_50gb_threshold"]}')

Manifest ensamblado:
  Fuentes: 7
  Peso total: 89.732 GB
  Archivos: 8,847
  Umbral ≥ 50 GB: True


## 4.2 Hash global del manifest y guardado en disco

Se calcula un hash MD5 del manifest completo (excluyendo el campo `manifest_hash` mismo) que sirve como huella única del dataset. Se guarda el archivo final en disco y se imprime un resumen estructurado para verificación.

In [9]:
manifest_sin_hash = json.dumps(manifest, sort_keys=True, ensure_ascii=False)
manifest_hash = hashlib.md5(manifest_sin_hash.encode('utf-8')).hexdigest()

manifest['manifest_hash_md5'] = manifest_hash

output_path = DIR_OUTPUT / 'manifest.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False, default=str)

file_size = output_path.stat().st_size

print(f'Manifest guardado en: {output_path.resolve()}')
print(f'Tamaño del archivo: {file_size:,} bytes ({file_size/1024:.1f} KB)')
print(f'Hash del manifest (MD5): {manifest_hash}')

print('\n=== Resumen final del dataset ===')
print(f'Dataset: {manifest["dataset_id"]} v{manifest["version"]}')
print(f'Repositorio: {manifest["repository"]["id"]}')
print(f'Fuentes: {manifest["summary"]["n_sources"]}')
print(f'Archivos totales: {manifest["summary"]["total_files"]:,}')
print(f'Peso total: {manifest["summary"]["total_size_gb"]} GB')
print(f'Cumple umbral 50 GB: {manifest["summary"]["meets_50gb_threshold"]}')
print(f'BBox espacial: {manifest["spatial_extent"]["bbox"]}')
print(f'Rango temporal proyecto: {manifest["temporal_extent"]["start"]} a {manifest["temporal_extent"]["end"]}')

print('\n=== Detalle por fuente ===')
for nombre, info in manifest['sources'].items():
    size_str = f'{info["size_bytes"]/1e9:.2f} GB' if info["size_bytes"] >= 1e9 else f'{info["size_bytes"]/1e6:.1f} MB'
    print(f'  {nombre:<12s} | {info["n_files"]:>5,d} archivos | {size_str:>9s} | hash: {info["aggregate_hash_md5"][:12]}...')

Manifest guardado en: C:\Users\Administrador\Desktop\github_proyecto03\proyecto-3\manifest\manifest_output\manifest.json
Tamaño del archivo: 13,786 bytes (13.5 KB)
Hash del manifest (MD5): 95fd112867f0d070a3294919f6907525

=== Resumen final del dataset ===
Dataset: geovision_cali_v1 v1.0
Repositorio: yeigen/fuentes-proyecto-3
Fuentes: 7
Archivos totales: 8,847
Peso total: 89.732 GB
Cumple umbral 50 GB: True
BBox espacial: [-76.6, 3.3, -76.4, 3.55]
Rango temporal proyecto: 2021-01-01 a 2025-12-31

=== Detalle por fuente ===
  S5P_NO2      |   216 archivos |   20.0 MB | hash: 63eafa192b8e...
  S5P_O3       |   152 archivos |   14.8 MB | hash: 324303428baa...
  S5P_SO2      |   280 archivos |    8.1 MB | hash: 2538a6806e87...
  S2           | 8,102 archivos |  89.67 GB | hash: d56b0a756016...
  ERA5         |    40 archivos |    4.7 MB | hash: db81e0f09a3c...
  MODIS_AOD    |    54 archivos |   16.0 MB | hash: b616dc60b932...
  DAGMA        |     3 archivos |    0.9 MB | hash: 9dcabec1d59

# Sección 5: Versionado en Git

## 5.1 Instrucciones manuales para subir el manifest al repositorio Git

El archivo `manifest.json` debe versionarse en el repositorio Git del proyecto (no en el bucket de HuggingFace). Esto cumple el requisito del proyecto de tener el manifest "versionado en Git" y permite trazabilidad de cambios al panel.

In [10]:
print('Para subir el manifest a Git, ejecuta en tu terminal:')
print()
print(f'  cp {output_path.resolve()} <ruta_del_repo_git>/manifest.json')
print(f'  cd <ruta_del_repo_git>')
print(f'  git add manifest.json')
print(f'  git commit -m "Add dataset manifest v{VERSION} ({manifest["summary"]["total_size_gb"]} GB, {manifest["summary"]["total_files"]:,} files)"')
print(f'  git push')
print()
print(f'El manifest queda en: {output_path.resolve()}')
print(f'Hash MD5 para verificación: {manifest_hash}')

Para subir el manifest a Git, ejecuta en tu terminal:

  cp C:\Users\Administrador\Desktop\github_proyecto03\proyecto-3\manifest\manifest_output\manifest.json <ruta_del_repo_git>/manifest.json
  cd <ruta_del_repo_git>
  git add manifest.json
  git commit -m "Add dataset manifest v1.0 (89.732 GB, 8,847 files)"
  git push

El manifest queda en: C:\Users\Administrador\Desktop\github_proyecto03\proyecto-3\manifest\manifest_output\manifest.json
Hash MD5 para verificación: 95fd112867f0d070a3294919f6907525
